In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Update this path to wherever you uploaded your 'insurance-rag' folder in your Drive
PROJECT_PATH = "/content/drive/MyDrive/falcon978/insurance-rag/Insurance-RAG-ebee0665cbfaff8f853fd92902a9bc652d4c742a"

# Change directory to the project root
os.chdir(PROJECT_PATH)
print(f"Current working directory: {os.getcwd()}")

In [ ]:
!pip install -r requirements.txt

# Sometimes Colab needs a specific nest-asyncio patch to run FastAPI/async tasks within Jupyter cells
!pip install nest-asyncio

In [ ]:
import os
import nest_asyncio
from google.colab import userdata

# Apply async patch for Colab environments
nest_asyncio.apply()

# Securely load API keys from Colab Secrets
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Ensure DeepEval uses the OpenAI key for the evaluator judge
os.environ["DEEPEVAL_JUDGE_MODEL"] = "gpt-4o"

print("Environment variables configured successfully.")

In [ ]:
import os
from config import settings
from rag_ingestion.pipeline import ExtractionPipeline

# Only run this if your ChromaDB is empty in the Colab environment
if not os.path.exists(settings.chroma_dir):
    print("Vector DB not found. Running extraction pipeline...")
    
    # Ingest Care Supreme
    ExtractionPipeline(
        pdf_path="notebooks/care-supreme.pdf",
        persist_dir=settings.chroma_dir,
        collection_name="care_docs"
    ).run()
    
    # Ingest HDFC Optima
    ExtractionPipeline(
        pdf_path="notebooks/optima-secure.pdf",
        persist_dir=settings.chroma_dir,
        collection_name="hdfc_docs"
    ).run()
    
    print("Ingestion complete.")
else:
    print("Vector DB found. Proceeding to evaluation.")

In [ ]:
# Run the complete suite: Recall, Precision, Faithfulness, Relevancy, and Custom Logic Adherence
!deepeval test run evaluations/test_cases/test_standard_rag_metrics.py --output-file eval_results.json

In [ ]:
print("--- RUNNING RERANKER IMPACT TEST ---")
!deepeval test run evaluations/test_cases/test_reranker_impact.py

print("\n--- RUNNING HYBRID VS SEMANTIC TEST ---")
!deepeval test run evaluations/test_cases/test_hybrid_vs_semantic.py

In [ ]:
import json
import pandas as pd
from IPython.display import display

# 1. Load the DeepEval JSON dump
with open('eval_results.json', 'r') as f:
    data = json.load(f)

# 2. Flatten the nested JSON into a tabular format
rows = []
for tc in data.get('testCases', []):
    row = {
        "Query": tc.get("input", ""),
        "Overall Status": "✅ Pass" if tc.get("success") else "❌ Fail",
        "Generated Response": tc.get("actualOutput", "")[:150] + "..." # Truncate for readability
    }
    
    # Extract individual metric scores
    for metric in tc.get('metrics', []):
        metric_name = metric['name']
        row[f"{metric_name} Score"] = metric['score']
        
        # Optional: capture the reasoning for failed metrics
        if not metric['success']:
            row[f"{metric_name} Error"] = metric['reason']
            
    rows.append(row)

df = pd.DataFrame(rows)

# 3. Create a "UI-ish" visualization using Pandas Styler
def style_status(val):
    if val == '✅ Pass':
        return 'background-color: #d4edda; color: #155724; font-weight: bold;'
    elif val == '❌ Fail':
        return 'background-color: #f8d7da; color: #721c24; font-weight: bold;'
    return ''

def style_scores(val):
    if isinstance(val, (int, float)):
        if val >= 0.8: return 'color: green;'
        if val < 0.5: return 'color: red;'
    return ''

# Apply the styling and render the interactive HTML table
styled_df = df.style\
    .applymap(style_status, subset=['Overall Status'])\
    .applymap(style_scores)\
    .set_properties(**{'text-align': 'left', 'max-width': '300px', 'white-space': 'pre-wrap'})\
    .set_caption("Insurance RAG Evaluation Results")

display(styled_df)